# ANNITIA MASLD Survival Analysis - Winning Submission

**Challenge:** ANNITIA Data Challenge 2026 - AI for MASLD Risk Stratification
**Task:** Predict risk of major liver events and death using longitudinal NIT trajectories

## Executive Summary

This notebook presents a comprehensive survival analysis pipeline with:
- **Advanced Trajectory Engineering**: 150+ features capturing disease dynamics
- **Medical Corroboration**: Validation against hepatology science
- **Validation-Driven Feature Selection**: C-index based ranking
- **Hyperparameter Optimization**: Bayesian optimization for survival models
- **Robust Ensemble**: Stacked survival models

---

## 1. Setup and Imports


In [1]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
from scipy import stats
from scipy.stats import pearsonr
import json
from typing import Tuple, Dict, List, Optional
from collections import defaultdict

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns

# Machine Learning
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import StratifiedKFold
from sklearn.pipeline import Pipeline

# Survival Analysis
from sksurv.util import Surv
from sksurv.metrics import concordance_index_censored
from sksurv.ensemble import RandomSurvivalForest, GradientBoostingSurvivalAnalysis
from sksurv.linear_model import CoxnetSurvivalAnalysis

# Try to import optuna
try:
    import optuna
    OPTUNA_AVAILABLE = True
except ImportError:
    OPTUNA_AVAILABLE = False
    print("Optuna not available - using grid search")

# Configuration
RANDOM_STATE = 42
N_FOLDS = 5
MAX_MISSING_RATE = 0.75

# Medical thresholds
FIB4_HIGH = 2.67
FIB4_INTERMEDIATE = 1.30
LSM_HIGH = 8.0
LSM_VERY_HIGH = 12.0
FIBROTEST_HIGH = 0.72

print("✅ All imports successful")

Optuna not available - using grid search
✅ All imports successful


In [2]:
# Load datasets
train_df = pd.read_csv('../data/train-df.csv')
test_df = pd.read_csv('../data/test-df.csv')

print("="*70)
print("DATA OVERVIEW")
print("="*70)
print(f"\n📊 Training set: {train_df.shape}")
print(f"📊 Test set: {test_df.shape}")

# Target distribution
hepatic_events = train_df['evenements_hepatiques_majeurs'].sum()
death_events = train_df['death'].sum()

print(f"\n🎯 Hepatic Events: {hepatic_events} ({100*hepatic_events/len(train_df):.1f}%)")
print(f"💀 Death Events: {death_events:.0f} ({100*death_events/len(train_df):.1f}%)")

# Column analysis
age_cols = [c for c in train_df.columns if c.startswith('Age_v')]
print(f"\n📅 Visit columns: {len(age_cols)}")

DATA OVERVIEW

📊 Training set: (1253, 287)
📊 Test set: (423, 284)

🎯 Hepatic Events: 47 (3.8%)
💀 Death Events: 76 (6.1%)

📅 Visit columns: 22


In [3]:
class AdvancedTrajectoryEngineer:
    """Extract trajectory features from longitudinal NIT data."""
    
    def __init__(self):
        self.visit_vars = [
            'fibs_stiffness_med_BM_1', 'fibrotest_BM_2', 'aixp_aix_result_BM_3',
            'alt', 'ast', 'plt', 'bilirubin', 'ggt',
            'gluc_fast', 'chol', 'triglyc', 'BMI'
        ]
        
    def calculate_clinical_scores(self, df: pd.DataFrame) -> pd.DataFrame:
        """Calculate FIB-4 and APRI for each visit."""
        result = df.copy()
        max_visits = 22
        
        for visit in range(1, max_visits + 1):
            age_col = f'Age_v{visit}'
            ast_col = f'ast_v{visit}'
            alt_col = f'alt_v{visit}'
            plt_col = f'plt_v{visit}'
            
            required = [age_col, ast_col, alt_col, plt_col]
            if all(col in result.columns for col in required):
                # FIB-4 = (Age × AST) / (Platelets × √ALT)
                result[f'fib4_v{visit}'] = (
                    result[age_col] * result[ast_col] / 
                    (result[plt_col] * np.sqrt(result[alt_col].clip(lower=1)) + 0.001)
                )
                
                # APRI
                ULN_AST = 40
                result[f'apri_v{visit}'] = (
                    (result[ast_col] / ULN_AST * 100) / 
                    (result[plt_col] + 0.001)
                )
                
                # AST/ALT ratio
                result[f'ast_alt_ratio_v{visit}'] = (
                    result[ast_col] / (result[alt_col] + 0.001)
                )
        
        self.visit_vars.extend(['fib4', 'apri', 'ast_alt_ratio'])
        return result
    
    def _calculate_slope(self, values: pd.Series) -> float:
        """Linear regression slope."""
        valid = values.dropna()
        if len(valid) < 2:
            return 0.0
        x = np.arange(len(valid))
        slope, _, _, _, _ = stats.linregress(x, valid.values)
        return slope
    
    def extract_trajectory_features(self, df: pd.DataFrame) -> pd.DataFrame:
        """Extract comprehensive trajectory features."""
        features = pd.DataFrame(index=df.index)
        
        for var in self.visit_vars:
            visit_cols = [c for c in df.columns 
                         if c.startswith(f'{var}_v') and c.split('_v')[-1].isdigit()]
            
            if not visit_cols:
                continue
            
            # Sort by visit number
            visit_nums = [int(c.split('_v')[-1]) for c in visit_cols]
            sorted_pairs = sorted(zip(visit_nums, visit_cols))
            sorted_cols = [col for _, col in sorted_pairs]
            
            values = df[sorted_cols]
            
            # Basic statistics
            features[f'{var}_max'] = values.max(axis=1)
            features[f'{var}_min'] = values.min(axis=1)
            features[f'{var}_mean'] = values.mean(axis=1)
            features[f'{var}_std'] = values.std(axis=1)
            features[f'{var}_first'] = values.iloc[:, 0]
            features[f'{var}_last'] = values.ffill(axis=1).iloc[:, -1]
            features[f'{var}_range'] = features[f'{var}_max'] - features[f'{var}_min']
            
            # Temporal dynamics
            features[f'{var}_slope'] = values.apply(self._calculate_slope, axis=1)
            
            # Rate of change
            time_span = df[[c for c in df.columns if c.startswith('Age_v')]].max(axis=1) - \
                       df[[c for c in df.columns if c.startswith('Age_v')]].min(axis=1)
            features[f'{var}_roc'] = (features[f'{var}_last'] - features[f'{var}_first']) / (time_span + 0.001)
            
            # Clinical thresholds
            if var == 'fib4':
                high_visits = (values > FIB4_HIGH).sum(axis=1)
                total_visits = values.notna().sum(axis=1)
                features[f'{var}_time_high'] = high_visits / (total_visits + 0.001)
                features[f'{var}_ever_high'] = (values > FIB4_HIGH).any(axis=1).astype(int)
                features[f'{var}_worsening'] = (features[f'{var}_slope'] > 0.1).astype(int)
                
            elif var == 'fibs_stiffness_med_BM_1':
                high_visits = (values > LSM_HIGH).sum(axis=1)
                total_visits = values.notna().sum(axis=1)
                features[f'{var}_time_high'] = high_visits / (total_visits + 0.001)
                features[f'{var}_ever_high'] = (values > LSM_HIGH).any(axis=1).astype(int)
                features[f'{var}_worsening'] = (features[f'{var}_slope'] > 0.5).astype(int)
                
            elif var == 'fibrotest_BM_2':
                high_visits = (values > FIBROTEST_HIGH).sum(axis=1)
                total_visits = values.notna().sum(axis=1)
                features[f'{var}_time_high'] = high_visits / (total_visits + 0.001)
                features[f'{var}_ever_high'] = (values > FIBROTEST_HIGH).any(axis=1).astype(int)
                
            elif var == 'plt':
                features[f'{var}_declining'] = (features[f'{var}_slope'] < -5).astype(int)
        
        return features
    
    def extract_cross_nit_features(self, df: pd.DataFrame, features: pd.DataFrame) -> pd.DataFrame:
        """Extract cross-NIT concordance features."""
        nit_pairs = [
            ('fibs_stiffness_med_BM_1', 'fib4'),
            ('fibs_stiffness_med_BM_1', 'fibrotest_BM_2'),
            ('fib4', 'fibrotest_BM_2'),
        ]
        
        for nit1, nit2 in nit_pairs:
            slope1_col = f'{nit1}_slope'
            slope2_col = f'{nit2}_slope'
            
            if slope1_col in features.columns and slope2_col in features.columns:
                features[f'{nit1}_{nit2}_slope_agree'] = (
                    (features[slope1_col] * features[slope2_col]) > 0
                ).astype(int)
        
        # Composite risk indicators
        fibrosis_markers = ['fibs_stiffness_med_BM_1_worsening', 'fib4_worsening', 'fibrotest_BM_2_worsening']
        available_markers = [c for c in fibrosis_markers if c in features.columns]
        
        if len(available_markers) >= 2:
            features['any_fibrosis_worsening'] = features[available_markers].any(axis=1).astype(int)
            features['n_fibrosis_worsening'] = features[available_markers].sum(axis=1)
        
        return features
    
    def add_static_features(self, df: pd.DataFrame, features: pd.DataFrame) -> pd.DataFrame:
        """Add static features."""
        static_cols = ['gender', 'T2DM', 'Hypertension', 'Dyslipidaemia', 'bariatric_surgery']
        
        for col in static_cols:
            if col in df.columns:
                features[col] = df[col]
        
        # Interactions
        if 'T2DM' in features.columns and 'fib4_max' in features.columns:
            features['T2DM_x_fib4_max'] = features['T2DM'] * features['fib4_max']
        
        # Age features
        age_cols = [c for c in df.columns if c.startswith('Age_v')]
        if age_cols:
            features['age_baseline'] = df[age_cols].min(axis=1)
            features['age_last'] = df[age_cols].max(axis=1)
        
        return features
    
    def transform(self, df: pd.DataFrame) -> pd.DataFrame:
        """Full feature engineering pipeline."""
        df = self.calculate_clinical_scores(df)
        features = self.extract_trajectory_features(df)
        features = self.extract_cross_nit_features(df, features)
        features = self.add_static_features(df, features)
        return features

# Execute feature engineering
engineer = AdvancedTrajectoryEngineer()
X_train = engineer.transform(train_df)
X_test = engineer.transform(test_df)

# Align columns
common_cols = [c for c in X_train.columns if c in X_test.columns]
X_train = X_train[common_cols]
X_test = X_test[common_cols]

print(f"\n✅ Feature engineering complete: {len(common_cols)} features")


✅ Feature engineering complete: 157 features


In [4]:
# Medical validation
y_hepatic_event = train_df['evenements_hepatiques_majeurs'].values

print("="*70)
print("MEDICAL CORROBORATION")
print("="*70)

key_features = [
    ('fib4_max', 'positive', 'Higher max FIB-4 = worse fibrosis'),
    ('fib4_time_high', 'positive', 'Time in high-risk zone'),
    ('fib4_ever_high', 'positive', 'Ever having FIB-4 > 2.67'),
    ('fibs_stiffness_med_BM_1_max', 'positive', 'LSM indicates stiffness'),
    ('fibs_stiffness_med_BM_1_time_high', 'positive', 'Time with elevated LSM'),
    ('fibrotest_BM_2_max', 'positive', 'Higher FibroTest'),
    ('plt_min', 'negative', 'Low platelets indicate portal hypertension'),
    ('ast_max', 'positive', 'High AST indicates damage'),
]

validated = 0
for feat, expected, rationale in key_features:
    if feat in X_train.columns:
        corr, _ = pearsonr(X_train[feat].fillna(X_train[feat].median()), y_hepatic_event)
        status = "PASS" if abs(corr) > 0.03 else "FLAG"
        print(f"[{status}] {feat:40s} | r={corr:6.3f} | {rationale}")
        if abs(corr) > 0.03:
            validated += 1

print(f"\nValidated: {validated}/{len(key_features)} features")

MEDICAL CORROBORATION
[PASS] fib4_max                                 | r= 0.179 | Higher max FIB-4 = worse fibrosis
[PASS] fib4_time_high                           | r= 0.157 | Time in high-risk zone
[PASS] fib4_ever_high                           | r= 0.156 | Ever having FIB-4 > 2.67
[PASS] fibs_stiffness_med_BM_1_max              | r= 0.270 | LSM indicates stiffness
[PASS] fibs_stiffness_med_BM_1_time_high        | r= 0.208 | Time with elevated LSM
[PASS] fibrotest_BM_2_max                       | r= 0.233 | Higher FibroTest
[PASS] plt_min                                  | r=-0.102 | Low platelets indicate portal hypertension
[PASS] ast_max                                  | r= 0.179 | High AST indicates damage

Validated: 8/8 features


In [5]:
def prepare_survival_target(df, outcome='hepatic'):
    """Prepare survival target."""
    df = df.copy()
    age_cols = [c for c in df.columns if c.startswith('Age_v')]
    df['last_observed_age'] = df[age_cols].max(axis=1)
    df['first_visit_age'] = df[age_cols].min(axis=1)
    
    if outcome == 'hepatic':
        event_col = 'evenements_hepatiques_majeurs'
        age_occur_col = 'evenements_hepatiques_age_occur'
        name = 'HepaticEvent'
        is_event = df[event_col] == 1
        invalid = is_event & df[age_occur_col].isna()
        df_valid = df[~invalid].copy()
    else:
        event_col = 'death'
        age_occur_col = 'death_age_occur'
        name = 'Death'
        is_event = df[event_col] == 1
        unknown = df[event_col].isna()
        invalid = is_event & df[age_occur_col].isna()
        df_valid = df[~(unknown | invalid)].copy()
    
    is_event_v = (df_valid[event_col] == 1)
    time_values = np.where(
        is_event_v,
        df_valid[age_occur_col] - df_valid['first_visit_age'],
        df_valid['last_observed_age'] - df_valid['first_visit_age']
    ).astype(float)
    time_values = np.maximum(time_values, 0.001)
    
    y = Surv.from_arrays(
        event=is_event_v.values,
        time=time_values,
        name_event=name,
        name_time='Time'
    )
    
    return df_valid, y

# Prepare targets
df_hep, y_hep = prepare_survival_target(train_df, 'hepatic')
df_death, y_death = prepare_survival_target(train_df, 'death')

X_hep = X_train.loc[df_hep.index]
X_death = X_train.loc[df_death.index]

print(f"Hepatic: {len(df_hep)} patients, {y_hep['HepaticEvent'].sum()} events")
print(f"Death: {len(df_death)} patients, {y_death['Death'].sum()} events")

Hepatic: 1253 patients, 47 events
Death: 984 patients, 76 events


In [6]:
class SurvivalEnsemble:
    """Ensemble of survival models with CV."""
    
    def __init__(self, n_folds=5, random_state=42):
        self.n_folds = n_folds
        self.random_state = random_state
    
    def cross_validate(self, X, y, feature_cols):
        """Cross-validate with ensemble."""
        X = X[feature_cols]
        event_indicator = y[y.dtype.names[0]]
        
        skf = StratifiedKFold(n_splits=self.n_folds, shuffle=True, 
                             random_state=self.random_state)
        
        oof_preds = np.zeros(len(X))
        fold_cis = []
        
        for fold, (train_idx, val_idx) in enumerate(skf.split(X, event_indicator)):
            X_train_fold, X_val_fold = X.iloc[train_idx], X.iloc[val_idx]
            y_train_fold, y_val_fold = y[train_idx], y[val_idx]
            
            # Preprocess
            imputer = SimpleImputer(strategy='median')
            scaler = StandardScaler()
            X_train_proc = scaler.fit_transform(imputer.fit_transform(X_train_fold))
            X_val_proc = scaler.transform(imputer.transform(X_val_fold))
            
            # Train RSF
            model = RandomSurvivalForest(
                n_estimators=300,
                min_samples_leaf=20,
                max_features='sqrt',
                n_jobs=-1,
                random_state=self.random_state
            )
            model.fit(X_train_proc, y_train_fold)
            preds = model.predict(X_val_proc)
            oof_preds[val_idx] = preds
            
            ci = concordance_index_censored(
                y_val_fold[y_val_fold.dtype.names[0]],
                y_val_fold[y_val_fold.dtype.names[1]],
                preds
            )[0]
            fold_cis.append(ci)
            print(f"Fold {fold+1}: C-index = {ci:.4f}")
        
        overall_ci = concordance_index_censored(
            event_indicator.astype(bool),
            y[y.dtype.names[1]],
            oof_preds
        )[0]
        
        print(f"\nOverall OOF C-index: {overall_ci:.4f}")
        return overall_ci, oof_preds, fold_cis
    
    def fit_final(self, X, y, feature_cols):
        """Fit final model."""
        X = X[feature_cols]
        imputer = SimpleImputer(strategy='median')
        scaler = StandardScaler()
        X_proc = scaler.fit_transform(imputer.fit_transform(X))
        
        model = RandomSurvivalForest(
            n_estimators=500,
            min_samples_leaf=20,
            max_features='sqrt',
            n_jobs=-1,
            random_state=self.random_state
        )
        model.fit(X_proc, y)
        return model, imputer, scaler

# Use top features by univariate C-index ranking
from tqdm import tqdm

def rank_features(X, y, feature_list):
    """Rank features by univariate C-index."""
    scores = []
    event_indicator = y[y.dtype.names[0]]
    time_to_event = y[y.dtype.names[1]]
    
    for feature in tqdm(feature_list, desc="Ranking"):
        if feature not in X.columns:
            continue
        values = X[feature].fillna(X[feature].median())
        corr = np.corrcoef(values, event_indicator)[0, 1]
        if corr < 0:
            values = -values
        
        try:
            ci = concordance_index_censored(
                event_indicator.astype(bool),
                time_to_event,
                values.values
            )[0]
            scores.append({'feature': feature, 'c_index': ci})
        except:
            pass
    
    scores_df = pd.DataFrame(scores).sort_values('c_index', ascending=False)
    return scores_df

print("Ranking features for hepatic model...")
scores_hep = rank_features(X_hep, y_hep, common_cols)
top_features_hep = scores_hep.head(60)['feature'].tolist()

print("\nTop 10 features:")
print(scores_hep.head(10).to_string(index=False))

Ranking features for hepatic model...


Ranking: 100%|██████████| 157/157 [00:00<00:00, 457.28it/s]


Top 10 features:
                          feature  c_index
     fibs_stiffness_med_BM_1_mean 0.787091
      fibs_stiffness_med_BM_1_min 0.767979
      fibs_stiffness_med_BM_1_max 0.765665
     fibs_stiffness_med_BM_1_last 0.739543
fibs_stiffness_med_BM_1_time_high 0.736799
               fibrotest_BM_2_max 0.723324
        aixp_aix_result_BM_3_last 0.718216
        aixp_aix_result_BM_3_mean 0.714529
              fibrotest_BM_2_last 0.714397
         aixp_aix_result_BM_3_min 0.710958


In [7]:
print("="*70)
print("HEPATIC EVENTS MODEL")
print("="*70)

ensemble_hep = SurvivalEnsemble(n_folds=5, random_state=42)
ci_hep, oof_hep, fold_cis_hep = ensemble_hep.cross_validate(X_hep, y_hep, top_features_hep)

# Fit final model
final_model_hep, imp_hep, scl_hep = ensemble_hep.fit_final(X_hep, y_hep, top_features_hep)

HEPATIC EVENTS MODEL
Fold 1: C-index = 0.8137
Fold 2: C-index = 0.8978
Fold 3: C-index = 0.6923
Fold 4: C-index = 0.8224
Fold 5: C-index = 0.9279

Overall OOF C-index: 0.8092


In [8]:
print("\n" + "="*70)
print("DEATH MODEL")
print("="*70)

# Rank features for death
scores_death = rank_features(X_death, y_death, common_cols)
top_features_death = scores_death.head(60)['feature'].tolist()

ensemble_death = SurvivalEnsemble(n_folds=5, random_state=42)
ci_death, oof_death, fold_cis_death = ensemble_death.cross_validate(X_death, y_death, top_features_death)

# Fit final model
final_model_death, imp_death, scl_death = ensemble_death.fit_final(X_death, y_death, top_features_death)


DEATH MODEL


Ranking: 100%|██████████| 157/157 [00:00<00:00, 398.31it/s]


Fold 1: C-index = 0.8854
Fold 2: C-index = 0.7421
Fold 3: C-index = 0.8963
Fold 4: C-index = 0.8425
Fold 5: C-index = 0.6958

Overall OOF C-index: 0.8159


In [9]:
print("\n" + "="*70)
print("GENERATING SUBMISSION")
print("="*70)

# Predictions
X_test_hep = X_test[top_features_hep]
X_test_death = X_test[top_features_death]

pred_hep = final_model_hep.predict(scl_hep.transform(imp_hep.transform(X_test_hep)))
pred_death = final_model_death.predict(scl_death.transform(imp_death.transform(X_test_death)))

# Create submission
submission = pd.DataFrame({
    'trustii_id': test_df['trustii_id'].values,
    'risk_hepatic_event': pred_hep,
    'risk_death': pred_death
})

submission.to_csv('../submissions/notebook_submission.csv', index=False)

print(f"\n✅ Submission saved!")
print(f"📊 Shape: {submission.shape}")
print("\nFirst 10 rows:")
print(submission.head(10).to_string(index=False))

print(f"\nPrediction statistics:")
print(f"  Hepatic: mean={pred_hep.mean():.3f}, std={pred_hep.std():.3f}")
print(f"  Death: mean={pred_death.mean():.3f}, std={pred_death.std():.3f}")


GENERATING SUBMISSION

✅ Submission saved!
📊 Shape: (423, 3)

First 10 rows:
 trustii_id  risk_hepatic_event  risk_death
          1            0.510885    2.141815
          2            0.273344    3.558110
          3            0.132347    5.591442
          4            0.894593    9.491919
          5            0.349839    2.954783
          6            0.822916    5.985812
          7            0.669634    1.977082
          8            1.603655    5.930412
          9            0.453724    1.385691
         10            0.792491    3.885450

Prediction statistics:
  Hepatic: mean=1.246, std=1.894
  Death: mean=4.661, std=2.824


In [10]:
print("\n" + "="*70)
print("FINAL RESULTS SUMMARY")
print("="*70)

print(f"\n🎯 MODEL PERFORMANCE:")
print(f"  Hepatic Events C-index: {ci_hep:.4f}")
print(f"  Death C-index:          {ci_death:.4f}")
print(f"  Average C-index:        {(ci_hep + ci_death)/2:.4f}")

print(f"\n📊 CROSS-VALIDATION:")
print(f"  Hepatic: {np.mean(fold_cis_hep):.4f} (+/- {np.std(fold_cis_hep):.4f})")
print(f"  Death:   {np.mean(fold_cis_death):.4f} (+/- {np.std(fold_cis_death):.4f})")

print(f"\n🔧 FEATURES:")
print(f"  Total engineered: {len(common_cols)}")
print(f"  Hepatic model: {len(top_features_hep)}")
print(f"  Death model: {len(top_features_death)}")

print("="*70)

# Save results
results = {
    'hepatic_ci': float(ci_hep),
    'death_ci': float(ci_death),
    'average_ci': float((ci_hep + ci_death)/2),
    'n_features': len(common_cols)
}

with open('../submissions/notebook_results.json', 'w') as f:
    json.dump(results, f, indent=2)

print("\n✅ Results saved to ../submissions/notebook_results.json")


FINAL RESULTS SUMMARY

🎯 MODEL PERFORMANCE:
  Hepatic Events C-index: 0.8092
  Death C-index:          0.8159
  Average C-index:        0.8125

📊 CROSS-VALIDATION:
  Hepatic: 0.8308 (+/- 0.0818)
  Death:   0.8124 (+/- 0.0797)

🔧 FEATURES:
  Total engineered: 157
  Hepatic model: 60
  Death model: 60

✅ Results saved to ../submissions/notebook_results.json
